In [ ]:
!pip install -q sentence-transformers faiss-cpu transformers accelerate

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Örnek İHA Teknik & Operasyon Kılavuzu (Chunks / Bilgi Parçaları)
# Gerçek hayatta 500 sayfalık bir PDF'i belirli karakter/kelime sınırlarıyla (örneğin 300 kelimelik parçalar ve 50 kelimelik örtüşme/overlap ile) böleriz.
dokuman_parcalari = [
    "BÖLÜM 1 - MOTOR VE İTKİ SİSTEMİ: KIZILELMA-X İHA platformunda ana itki sistemi maksimum 115 derece egzoz gazı sıcaklığında sürekli çalışabilir. Sıcaklık 120 dereceyi aşarsa otomatik güç kısma (derating) modu devreye girer.",
    "BÖLÜM 2 - HABERLEŞME VE DATALINK: Yer kontrol istasyonu (YKİ) ile görüş hattı (LOS) haberleşmesi C-Bant üzerinden sağlanır. Sinyal kaybı 15 saniyeyi aşarsa İHA otomatik olarak 'Eve Dönüş' (RTH - Return to Home) prosedürünü başlatır.",
    "BÖLÜM 3 - BATARYA VE GÜÇ YÖNETİMİ: Ana aviyonik batarya voltajı 22.4V altına düştüğünde kritik düşük voltaj uyarısı verilir. Bu durumda faydalı yük (kamera/radar) gücü kesilerek yalnızca uçuş kontrol bilgisayarına enerji aktarılır.",
    "BÖLÜM 4 - İNİŞ TAKIMLARI VE FRENLEME: Hidrolik iniş takımları yerden 150 metre irtifada açılır. İniş sonrası karbon-seramik frenlerin soğuma süresi minimum 20 dakikadır; bu süre dolmadan yeni kalkış izni verilmez.",
    "BÖLÜM 5 - BUZLANMA ÖNLEME SİSTEMİ: Kanat hücum kenarlarındaki elektro-termal buz çözücü sistem, dış hava sıcaklığı +2 derecenin altına düştüğünde ve nem oranı %80'i geçtiğinde otonom olarak aktifleşir."
]

# 2. Türkçe destekli çok dilli Embedding modelini yüklüyoruz
print("Embedding modeli yükleniyor...")
embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# 3. Doküman parçalarını sayısal vektörlere (Embedding) dönüştürüyoruz
dokuman_vektorleri = embed_model.encode(dokuman_parcalari)

print(f"Toplam parça sayısı: {dokuman_vektorleri.shape[0]}")
print(f"Her bir parçanın vektör boyutu (kaç sayıdan oluştuğu): {dokuman_vektorleri.shape[1]}")
print(f"İlk parçanın ilk 5 sayısı: {dokuman_vektorleri[0][:5]}")

# 4. FAISS Vektör Veri Tabanı oluşturma ve vektörleri içine ekleme
vektor_boyutu = dokuman_vektorleri.shape[1]  # 384 boyut
index = faiss.IndexFlatL2(vektor_boyutu)    # Öklid mesafesi tabanlı arama indeksi
index.add(np.array(dokuman_vektorleri).astype("float32"))

print("Vektör veri tabanı (FAISS) başarıyla oluşturuldu!")

Embedding modeli yükleniyor...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Toplam parça sayısı: 5
Her bir parçanın vektör boyutu (kaç sayıdan oluştuğu): 384
İlk parçanın ilk 5 sayısı: [0.00277909 0.28100023 0.0117034  0.08953051 0.01050485]
Vektör veri tabanı (FAISS) başarıyla oluşturuldu!


In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# 1. Daha güçlü çok dilli embedding modeli (T4 GPU dostu)
embed_model_id = "intfloat/multilingual-e5-base"
embed_model = SentenceTransformer(embed_model_id)

# Örnek teknik doküman parçaları (Aynı metinler)
chunks = [
    (
        "Bölüm 1 - Uçuş Kontrol Sistemi: Otopilot modunda maksimum irtifa 15.000"
        " feet olarak sınırlandırılmıştır. Rüzgar hızı 35 knot üzerine çıkarsa"
        " sistem otomatik olarak eve dönüş (RTL) moduna geçer."
    ),
    (
        "Bölüm 2 - Haberleşme ve Datalink: Yer kontrol istasyonu ile görüş"
        " hattı (LOS) haberleşmesi 150 km menzile sahiptir. Sinyal kaybı 10"
        " saniyeden uzun sürerse hava aracı önceden tanımlı acil durum"
        " rotasına yönelir."
    ),
    (
        "Bölüm 3 - Batarya ve Güç Yönetimi: Lityum-Polimer batarya paketleri"
        " tam şarjda 4.5 saat uçuş süresi sağlar. Batarya voltajı 22.4V altına"
        " düştüğünde kritik seviye uyarısı verilir ve faydalı yük (kamera)"
        " gücü kesilir."
    ),
]

# 2. E5 modeli için dokümanların başına 'passage: ' öneki ekleyip normalize ediyoruz
passages = [f"passage: {chunk}" for chunk in chunks]
embeddings = embed_model.encode(passages, normalize_embeddings=True)

# 3. Kosinüs Benzerliği için IndexFlatIP (Inner Product) kullanıyoruz
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(np.array(embeddings, dtype="float32"))

# Test Sorgusu
query = "Yerle bağlantı koparsa uçak ne yapar?"
query_embedding = embed_model.encode(
    [f"query: {query}"], normalize_embeddings=True
)

# En yakın 2 dokümanı getir
scores, indices = index.search(np.array(query_embedding, dtype="float32"), k=2)

print(f"Sorgu: {query}\n" + "-" * 50)
for rank, (idx, score) in enumerate(zip(indices[0], scores[0]), start=1):
  print(f"{rank}. Eşleşme (Benzerlik Skoru: {score:.4f}):")
  print(f"   {chunks[idx]}\n")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

... (5 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Sorgu: Yerle bağlantı koparsa uçak ne yapar?
--------------------------------------------------
1. Eşleşme (Benzerlik Skoru: 0.8130):
   Bölüm 1 - Uçuş Kontrol Sistemi: Otopilot modunda maksimum irtifa 15.000 feet olarak sınırlandırılmıştır. Rüzgar hızı 35 knot üzerine çıkarsa sistem otomatik olarak eve dönüş (RTL) moduna geçer.

2. Eşleşme (Benzerlik Skoru: 0.8070):
   Bölüm 2 - Haberleşme ve Datalink: Yer kontrol istasyonu ile görüş hattı (LOS) haberleşmesi 150 km menzile sahiptir. Sinyal kaybı 10 saniyeden uzun sürerse hava aracı önceden tanımlı acil durum rotasına yönelir.



In [ ]:
!pip install -q -U bitsandbytes accelerate

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 4-bit kuantizasyon ayarı (T4 GPU'da ~5.5 GB VRAM kaplar, rahat çalışır)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

llm_id = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(llm_id)
model = AutoModelForCausalLM.from_pretrained(
    llm_id,
    quantization_config=bnb_config,
    device_map="auto"
)

def ask_rag(question, top_k=2):
    # 1. Retrieval (E5 formatıyla arama)
    q_emb = embed_model.encode([f"query: {question}"], normalize_embeddings=True)
    scores, indices = index.search(np.array(q_emb, dtype="float32"), k=top_k)

    retrieved_docs = [chunks[i] for i in indices[0]]
    context = "\n\n".join([f"[Kaynak {idx+1}]: {doc}" for idx, doc in enumerate(retrieved_docs)])

    # 2. Prompt Mühendisliği (Sıkı Grounding + Teknik Eş anlamlılık yorumlama)
    messages = [
        {
            "role": "system",
            "content": (
                "Sen bir İHA sistem mühendisliği asistanısın. Sadece sana verilen [Kaynak] metinlerindeki "
                "bilgileri kullanarak soruyu yanıtla. Metindeki teknik ifadeleri (örneğin voltaj düşüşü = pil azalması, "
                "sinyal kaybı = bağlantı kopması) doğru yorumla. Kaynaklarda hiç geçmeyen bir bilgi sorulursa "
                "'Kılavuzda bu bilgi yer almıyor.' de."
            )
        },
        {
            "role": "user",
            "content": f"Bağlam Bilgileri:\n{context}\n\nSoru: {question}"
        }
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,  # Halüsinasyonu önlemek için düşük sıcaklık
            do_sample=True
        )

    # Sadece üretilen yeni token'ları al
    output_ids = generated_ids[0][len(inputs.input_ids[0]):]
    answer = tokenizer.decode(output_ids, skip_special_tokens=True)
    return answer, retrieved_docs

# İki kritik soruyu da test edelim
test_questions = [
    "Yerle bağlantı koparsa uçak ne yapar?",
    "Pil azalırsa kamera çalışmaya devam eder mi?"
]

for q in test_questions:
    ans, docs = ask_rag(q, top_k=2)
    print(f"SORU: {q}")
    print(f"CEVAP: {ans}")
    print("=" * 60)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

SORU: Yerle bağlantı koparsa uçak ne yapar?
CEVAP: Yer kontrol istasyonu ile bağlantı koparsa, hava aracı önceden tanımlı acil durum rotasına yönelir. Bu durumda sinyal kaybı 10 saniyeden uzun sürerse, uçağın bu eylemi gerçekleştirmesi gerekmektedir.
SORU: Pil azalırsa kamera çalışmaya devam eder mi?
CEVAP: Hayır, pil azaldığında kamera çalışmaya devam etmeyecektir. Kaynak 1'de belirtilen gibi, batarya voltajı 22.4V altına düştüğünde faydalı yük (kamera) gücü kesilir. Bu, batarya şarj seviyesinin kritik bir noktaya ulaştığını gösterir ve bu durumda kamera etkili bir şekilde çalışamaz.
